# What Climate Change Risks Are?

We could have began with a long and boring introduction about climate change risks, making a long lists about climate change issues. Basically, we could read one of the many report about it, like the one from the **Task-Force on Climate-Related Financial Disclosures** (TCFD). 

But let's take another option, and leverage on new AI capabilities to help us understand better the topic. In this first lesson, we will try to understand a bit better what climate change risks are by using **Natural Language Processing** (NLP).

To do so, we want to **extract keywords characterizing climate change risks** issues. We will learn how to extract a set of keywords describing climate risks from TCFD reports. 


When we want to understand key information from specific documents, we turn towards **keyword extraction**. It corresponds to the automated process of extracting the words and phrases that are most relevant.

To do so, we prefer to work with **semantic similarity** than **statistical properties** of a text (see Rake and YAKE! methods for statistical-based keywords extraction).

We will rely on **BERT** capabilities here. BERT allows us to transform phrases and documents to vectors that capture their meaning.

We will first try to use BERT to create our own **keyword extraction model**, in order to better undestand the process, then briefly present the **KeyBERT** library that we will use thereafter.


## Leveraging on BERT capabilities to extract climate change risks keywords

In this part, we will try to build upon BERT capabilities in order to create our own keyword extraction model. This will allows us to better understand the process and potential caveats.

### Data 

Let's use a simple paragraph that will served us as a document, extracted from a TCFD report:

In [ ]:
doc = """
Physical risks resulting from climate change can be event driven (acute) or longer-term shifts 
(chronic) in climate patterns. Physical risks may have financial implications for organizations, such 
as direct damage to assets and indirect impacts from supply chain disruption. Organizations’ 
financial performance may also be affected by changes in water availability, sourcing, and quality; 
food security; and extreme temperature changes affecting organizations’ premises, operations, 
supply chain, transport needs, and employee safety.
"""

### Candidate Keywords / Keyphrases

We will start by creating a list of candidate keywords or keyphrases from a document. We keep it simple by using **scikitlearn** CountVectorizer method. This allows to specify the length of the keywords and make them into keyphrases. We can also quickly remove stop words with that method.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

n_gram_range = (1, 1)
stop_words = "english"

# Extract candidate words / phrases
def extract_candidate_words(n_gram_range, stop_words):
  count = CountVectorizer(ngram_range = n_gram_range, stop_words = stop_words).fit([doc])
  candidates = count.get_feature_names_out()
  return candidates 

candidates = extract_candidate_words(n_gram_range = n_gram_range, stop_words = stop_words)
print(candidates)

['acute' 'affected' 'affecting' 'assets' 'availability' 'chain' 'change'
 'changes' 'chronic' 'climate' 'damage' 'direct' 'disruption' 'driven'
 'employee' 'event' 'extreme' 'financial' 'food' 'impacts' 'implications'
 'indirect' 'longer' 'needs' 'operations' 'organizations' 'patterns'
 'performance' 'physical' 'premises' 'quality' 'resulting' 'risks'
 'safety' 'security' 'shifts' 'sourcing' 'supply' 'temperature' 'term'
 'transport' 'water']


We can already see some interesting terms that appear. We can further use n_gram_range to change the size of the resulting candidates. For example, we could set it to (3,3) to get candidates phrases that include 3 keywords. Let's play with it to get a sense of the impact on candidates!

### Embeddings

The next step is to convert the **document** and the **candidate keywords / keyphrases** to **numerical data**: this is done through a process called **embedding**.
We use BERT to do so, as it has shown great results for both similarity and paraphrasing tasks.

We need to install the package **sentence-transformers** that will allow us to create embeddings:

In [ ]:
!pip install sentence-transformers # to install the package sentence-transformer

Now we are running the following code to transform the document and candidates into vectors:

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('distilbert-base-nli-mean-tokens') # we use Distilbert, a variant of BERT which proved to be performant and lighter!

def get_embeddings(model, doc, candidates):
  doc_embedding = model.encode([doc])
  candidate_embeddings = model.encode(candidates)
  return doc_embedding, candidate_embeddings 

doc_embedding, candidate_embeddings = get_embeddings(model = model, doc = doc, candidates = candidates)
print(candidate_embeddings)

[[-0.68191075 -0.87059385  0.05186342 ... -0.01992879  0.301174
  -0.9494943 ]
 [-0.40901613 -0.199091   -0.21486177 ... -0.63438     0.05151439
  -0.24857354]
 [-0.44906166 -0.21071814 -0.12306113 ... -0.5263949  -0.06260508
  -0.3927536 ]
 ...
 [-0.33074    -0.03926666  0.30435297 ... -0.1757524  -0.15117063
   0.0483488 ]
 [-0.7669608   0.11929482  0.23707195 ... -0.06826131 -0.10063779
  -0.4379287 ]
 [-0.34256718 -0.0058112   0.7193823  ... -0.28569353 -0.84701115
   0.6572497 ]]


Be aware that transformers models have a token limit. So if you put too large documents as an input, you will run into some errors. In that case, we need to consider to split up our document into paragraphs and mean pooling (taking the average) of resulting vectors.

### Cosine Similarity

In this final step, we need to **find the candidates that are most similar to the document**. We assume that the most similar candidates to the document are good keywords / keyphrases for representing the document. 

To calculate the similarity between candidates and the document, we will be using the **cosine similarity** (basically the dot product) between vectors: 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity 

def get_keywords(top_n, doc_embedding, candidate_embeddings):
  distances = cosine_similarity(doc_embedding, candidate_embeddings)
  keywords = [candidates[index] for index in distances.argsort()[0][-top_n:]]
  return keywords

keywords = get_keywords(top_n = 5, doc_embedding = doc_embedding, candidate_embeddings = candidate_embeddings)
print(keywords)

['disruption', 'affected', 'affecting', 'impacts', 'climate']


And that's it! Those are the top 5 most similar candidates to the input document and the resulting keywords. 
How does those keywords look like? Does they seem to describe a document about climate risks? 
Let's try to see what happens with n_gram_range changes to (3,3):

In [ ]:
candidates = extract_candidate_words(n_gram_range = (3,3), stop_words = stop_words)
doc_embedding, candidate_embeddings = get_embeddings(model = model, doc = doc, candidates = candidates)
keywords = get_keywords(top_n = 5, doc_embedding = doc_embedding, candidate_embeddings = candidate_embeddings)
print(keywords)

['shifts chronic climate', 'chronic climate patterns', 'risks resulting climate', 'climate change event', 'resulting climate change']


Does it seems better? It seems that we have a lot of repetition of the word 'climate'. Maybe should we aim for more diversity in our keywords / keyphrases.

### Diversification 

There is a reason why similar results are returned...they best represent the document! If we were to diversify the keywords / keyphrases then they are less likely to represent the document well as a **collective**. 

So, the diversification of our results requires a delicate balance between the accuracy of keywords / keyphrases and the diversity between them. 

We can achieve that with two algorithms:
- Max Sum Similarity
- Maximal Marginal Relevance

#### Max Sum Similarity

The maximum sum distance between pairs of data is defined as the pairs of data for which the distance between them is maximized. In our case, we want to maximize the candidate similarity to the document while minimizing the similarity between candidates. 

To do so, we select the top 20 keywords / keyphrases, and from those 20, select the 5 that are the least similar to each other:

In [ ]:
import numpy as np
import itertools 

def max_sum_sim(doc_embedding, word_embeddings, words, top_n, nr_candidates):
  # calculate the distances and extract keywords 
  distances = cosine_similarity(doc_embedding, candidate_embeddings)
  distances_candidates = cosine_similarity(candidate_embeddings, candidate_embeddings)

  # get the top_n words as candidates based on consine similarity 
  words_idx = list(distances.argsort()[0][-nr_candidates:])
  words_vals = [candidates[index] for index in words_idx]
  distanes_candidates = distances_candidates[np.ix_(words_idx, words_idx)]

  # calculate the combination of words that are the least similar to each other 
  min_sim = np.inf
  candidate = None 
  for combination in itertools.combinations(range(len(words_idx)), top_n):
    sim = sum([distances_candidates[i][j] for i in combination for j in combination if i != j])
    if sim < min_sim:
      candidate = combination 
      min_sim = sim 
  

  return [words_vals[idx] for idx in candidate]

keywords = max_sum_sim(doc_embedding = doc_embedding, 
                       word_embeddings = candidate_embeddings, 
                       words = candidates, 
                       top_n = 5, nr_candidates = 10)
print(keywords)

['financial implications organizations', 'risks financial implications', 'changes affecting organizations', 'chronic climate patterns', 'resulting climate change']


A low nr_candidates will results in similary keyphrases to the original cosine similarity methods, but high nr_candidates will create more diverse keyphrases:

In [ ]:
keywords = max_sum_sim(doc_embedding = doc_embedding, 
                       word_embeddings = candidate_embeddings, 
                       words = candidates, 
                       top_n = 5, nr_candidates = 20)
print(keywords)

['disruption organizations financial', 'supply chain disruption', 'financial performance affected', 'shifts chronic climate', 'chronic climate patterns']


There is a tradeoff between accuracy and diversity that you must keep in mind. If we increase the nr_candidates, then there is a good change that you get diverse keywords but that are not very good representations of the document.
A good rule of thumb can be to keep nr_candidates less thant 20% of the total number of unique words in the document.

#### Maximal Marginal Relevance 

The other method for diversifying our results is the Maximal Marginal Relevance (MMR). It tries to minimize redundancy and maximize the diversity of results in text summarization tasks. Fortunately, a keyword extraction algorithm called EmbedRank has implemented a version of MMR that allows us to use it for diversifying our keyword / keyphrases.

We start by selecting the keyword / keyphrase that is the most similar to the document. Then, we iteratively select new candidates that are both similar to the document and not similar to the already selected keyword / keyphrases:

In [ ]:
import numpy as np

def mmr(doc_embedding, word_embeddings, words, top_n, diversity):
  # Extract similarity within words, and between words and the document 
  word_doc_similarity = cosine_similarity(word_embeddings, doc_embedding)
  word_similarity = cosine_similarity(word_embeddings)

  # Initializes candidates and already choose best keywrod / keyphrases 
  keywords_idx = [np.argmax(word_doc_similarity)]
  candidates_idx = [i for i in range(len(words)) if i != keywords_idx[0]]

  for _ in range(top_n -1):
    # Extract similarities within candidates and between candidates and selected keywords 
    candidate_similarities = word_doc_similarity[candidates_idx, :]
    target_similarities = np.max(word_similarity[candidates_idx][:, keywords_idx], axis = 1)

    #calculate MMR 
    mmr = (1 - diversity) * candidate_similarities - diversity * target_similarities.reshape(-1, 1)
    mmr_idx = candidates_idx[np.argmax(mmr)]

    # update keywords & candidates 
    keywords_idx.append(mmr_idx)
    candidates_idx.remove(mmr_idx)

  return [words[idx] for idx in keywords_idx]

keywords = mmr(doc_embedding = doc_embedding,
               word_embeddings = candidate_embeddings, 
               words = candidates,
               top_n = 5,
               diversity = 0.2)
print(keywords)

['resulting climate change', 'risks resulting climate', 'climate change event', 'changes affecting organizations', 'chronic climate patterns']


Again, as you increase the diversity parameter, you will get more diverse keyphrases:

In [ ]:
keywords = mmr(doc_embedding = doc_embedding,
               word_embeddings = candidate_embeddings, 
               words = candidates,
               top_n = 5,
               diversity = 0.7)
print(keywords)

['resulting climate change', 'quality food security', 'organizations financial performance', 'physical risks resulting', 'transport needs employee']


## KeyBERT

KeyBERT is a fantastic Python package that will allow you to do all this process is three lines of code!**texte en gras**

In [ ]:
!pip install keybert #for installation purposes

Let's take again our previous example from the TCFD:

### Basic Usage

In [ ]:
from keybert import KeyBERT

doc = """
Physical risks resulting from climate change can be event driven (acute) or longer-term shifts 
(chronic) in climate patterns. Physical risks may have financial implications for organizations, such 
as direct damage to assets and indirect impacts from supply chain disruption. Organizations’ 
financial performance may also be affected by changes in water availability, sourcing, and quality; 
food security; and extreme temperature changes affecting organizations’ premises, operations, 
supply chain, transport needs, and employee safety.
"""

kw_model = KeyBERT()
keywords = kw_model.extract_keywords(doc)


In [ ]:
print(keywords)

[('risks', 0.4752), ('climate', 0.4596), ('impacts', 0.3638), ('organizations', 0.3432), ('disruption', 0.3107)]


We can set the parameter keyphrase_ngra_range to have keyphrases of 3 words, and also add a stop_words processing with english:

In [ ]:
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range = (1,3), stop_words = 'english')
print(keywords)

[('risks resulting climate', 0.6845), ('physical risks financial', 0.6005), ('physical risks resulting', 0.5805), ('risks financial implications', 0.5754), ('financial implications organizations', 0.5595)]


And we can highlith the keywords in the document by setting highlight:

In [ ]:
keywords = kw_model.extract_keywords(doc, highlight = True)

Physical risks resulting from climate change can be event driven acute or longer term shifts chronic in climate 
patterns Physical risks may have financial implications for organizations such as direct damage to assets and 
indirect impacts from supply chain disruption Organizations financial performance may also be affected by changes 
in water availability sourcing and quality food security and extreme temperature changes affecting organizations 
premises operations supply chain transport needs and employee safety

### Diversification 

As in our customized implementation, we can use either the Max Sum Distance method or the Maximal Marginal Relevance to get more diverse results.

#### Max Sum Distance 

Here it is:

In [ ]:
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range = (3,3), stop_words = 'english',
                          use_maxsum = True, nr_candidates = 20, top_n = 5)
print(keywords)

[('extreme temperature changes', 0.4332), ('chain disruption organizations', 0.4459), ('organizations financial performance', 0.4511), ('chronic climate patterns', 0.4697), ('physical risks resulting', 0.5805)]


#### Maximal Marginal Relevance 

And we can also use the MMR:

In [ ]:
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range = (3,3), stop_words = 'english',
                                     use_mmr = True, diversity = 0.7)
print(keywords)

[('risks resulting climate', 0.6845), ('organizations financial performance', 0.4511), ('water availability sourcing', 0.2585), ('term shifts chronic', 0.2318), ('transport needs employee', 0.2156)]


### Embedding Models 

We can choose the embedding models that we want to use. For example, we can process as before:

In [ ]:
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer 

sentence_model = SentenceTransformer('distilbert-base-nli-mean-tokens')
kw_model = KeyBERT(model = sentence_model)

## Let's Go Bigger!



In [ ]:
!pip install PyPDF2

In [ ]:
!pip install pdfplumber

In [ ]:
import pdfplumber
pdf = pdfplumber.open('FINAL-2017-TCFD-Report-11052018.pdf')

from functools import partial
def not_within_bboxes(obj, bboxes):
  def obj_in_bbox(_bbox):
    """Find Bboxes of objexts."""
    v_mid = (obj["top"] + obj["bottom"]) / 2
    h_mid = (obj["x0"] + obj["x1"]) / 2
    x0, top, x1, bottom = _bbox
    return (h_mid >= x0) and (h_mid < x1) and (v_mid >= top) and (v_mid < bottom)
  return not any(obj_in_bbox(__bbox) for __bbox in bboxes)

def extract(page):
  """Extract PDF text, Filter tables and delete in-par breaks."""
  # Filter-out tables
  if page.find_tables() != []:
    # Get the bounding boxes of the tables on the page.
    bboxes = [table.bbox for table in page.find_tables()]
    bbox_not_within_bboxes = partial(not_within_bboxes, bboxes=bboxes)
    # Filter-out tables from page
    page = page.filter(bbox_not_within_bboxes)
  # Extract Text
  extracted = page.extract_text()
  # Delete in-paragraph line breaks
  extracted = extracted.replace(".\n", "**/m" # keep par breaks
                      ).replace(". \n", "**/m" # keep par breaks
                      ).replace("\n", "" # delete in-par breaks
                      ).replace("**/m", ".\n\n") # restore par break
  return extracted

text = extract(pdf.pages[3])
print(text)


 ranging from $4.2 trillion to $43 trillion between now and the end of the century.2 The study highlights that “much of the impact on future assets will come through weaker growth and lower asset returns across the board.” This suggests investors may not be able to avoid climate-related risks by moving out of certain asset classes as a wide range of asset types could be affected. Both investors and the organizations in which they invest, therefore, should consider their longer-term strategies and most efficient allocation of capital. Organizations that invest in activities that may not be viable in the longer term may be less resilient to the transition to a lower-carbon economy; and their investors will likely experience lower returns. Compounding the effect on longer-term returns is the risk that present valuations do not adequately factor in climate-related risks because of insufficient information. As such, long-term investors need adequate information on how organizations are prep

In [ ]:
import PyPDF2

#Define path to PDF file
pdf_file_name = 'FINAL-2017-TCFD-Report-11052018.pdf'

reader = PyPDF2.PdfFileReader(pdf_file_name)
page = reader.getPage(3)
text = page.extractText()
print(text)

 
Recommendations  of the Task Force on Clima te-related Financial Disclosures  iii ranging from $4.2 trillion to $43 trillion between now and the end of the century .2 The study 
highlights that “much of the impact on future assets will come through weaker growth and lower  
asset returns across the board. ” This suggests  investors may  not be able to avoid climate -related 
risks by moving out of certain  asset classes as a wide range of asset types could be affected . Both 
investors and the organizations in which they invest, therefore, should  consider  their longer -term 
strategies and most  efficient allocation of capital. Organizations that invest  in activities that may  
not be viable in the longer  term  may  be less resilient to the transition to a low er-carbon economy ; 
and their investors will likely experience lower  returns. Compounding the  effect on long er-term 
returns is the risk that present valuations do n ot adequately factor in climate -related risks 
becau

In [ ]:
import nltk
nltk.download('punkt')
a_list = nltk.tokenize.sent_tokenize(text)
print(len(a_list))
print(a_list)

19
[' ranging from $4.2 trillion to $43 trillion between now and the end of the century.2 The study highlights that “much of the impact on future assets will come through weaker growth and lower asset returns across the board.” This suggests investors may not be able to avoid climate-related risks by moving out of certain asset classes as a wide range of asset types could be affected.', 'Both investors and the organizations in which they invest, therefore, should consider their longer-term strategies and most efficient allocation of capital.', 'Organizations that invest in activities that may not be viable in the longer term may be less resilient to the transition to a lower-carbon economy; and their investors will likely experience lower returns.', 'Compounding the effect on longer-term returns is the risk that present valuations do not adequately factor in climate-related risks because of insufficient information.', 'As such, long-term investors need adequate information on how organ

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
